<a href="https://colab.research.google.com/github/AcostaAlex10/hackathon-kit-Acosta-Borges-Pelinski/blob/Acosta/%20%20notebooks/eda_y_benchmark_vinculo_con_drive.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EDA y benchmark multimodelo — Desafío Machine Health

Sesión de la mañana del jueves. Objetivo: establecer el **techo de rendimiento
con datos exclusivamente tabulares**, antes de invertir la tarde en las señales
crudas.

Alcance de este notebook, deliberadamente acotado:

- Sólo `train_sup.parquet` y `test.parquet`. **El procesamiento de NPZ queda
  fuera** de esta etapa.
- Sólo la métrica oficial (`scoring.compute_score`). No se decide nada por
  accuracy ni logloss: la métrica es compuesta y no correlaciona con ellas.
- Validación `StratifiedGroupKFold(5)` agrupada por `machine_id`, sin excepción.

Contexto y hallazgos previos en `estrategia1_Borges.md`. Los tres que gobiernan
este notebook:

1. **No es multilabel.** `is_combo` vale 0 en las 1750 filas etiquetadas: cada
   ventana tiene exactamente 0 o 1 falla. Se modela como softmax de 14 estados.
2. **Train y test no comparten ninguna máquina.** De ahí la normalización por
   `machine_id` de la sección 2 y el agrupamiento de la validación.
3. **La capa de decisión de la métrica ya es óptima.** No se afilan ni se
   distorsionan las probabilidades: sólo se predice mejor.

## 0. Entorno

In [ ]:
import requests
import zipfile
import io
import os

# URL de descarga directa del archivo ZIP de Google Drive
# (Necesitas obtener la URL de descarga directa, el formato puede variar)
# Por ejemplo: 'https://drive.google.com/uc?export=download&id=YOUR_FILE_ID'
# Reemplaza 'YOUR_FILE_ID' con el ID del archivo en tu enlace.
# Para el enlace proporcionado 'https://drive.google.com/uc?id=1FKuxVH--rRJiK8tzfT_T8BW1WVj8v6wV'
# el ID del archivo es '1FKuxVH--rRJiK8tzfT_T8BW1WVj8v6wV'

# La URL proporcionada ya es un enlace de descarga directa de Google Drive.
zip_file_url = 'https://drive.google.com/uc?id=1FKuxVH--rRJiK8tzfT_T8BW1WVj8v6wV'
output_dir = './data_from_drive/' # Directorio donde se extraerán los archivos

print(f"Descargando archivo ZIP desde: {zip_file_url}")
response = requests.get(zip_file_url)
response.raise_for_status() # Lanza una excepción si la descarga falla

# Crear el directorio de salida si no existe
os.makedirs(output_dir, exist_ok=True)

# Descomprimir el archivo en el directorio especificado
with zipfile.ZipFile(io.BytesIO(response.content)) as zf:
    zf.extractall(output_dir)

print(f"Archivos descomprimidos en: {output_dir}")
print("Listado de archivos extraídos:")
for root, dirs, files in os.walk(output_dir):
    for name in files:
        print(os.path.join(root, name))
    for name in dirs:
        print(os.path.join(root, name) + '/')


Descargando archivo ZIP desde: https://drive.google.com/uc?id=1FKuxVH--rRJiK8tzfT_T8BW1WVj8v6wV
Archivos descomprimidos en: ./data_from_drive/
Listado de archivos extraídos:
./data_from_drive/participant_kit/
./data_from_drive/participant_kit/requirements.txt
./data_from_drive/participant_kit/baseline.ipynb
./data_from_drive/participant_kit/scoring.py
./data_from_drive/participant_kit/convert_raw_to_csv.py
./data_from_drive/participant_kit/baseline.py
./data_from_drive/participant_kit/__init__.py
./data_from_drive/participant_kit/eda_intro.ipynb
./data_from_drive/participant_kit/README.md
./data_from_drive/participant_kit/outputs/
./data_from_drive/participant_kit/outputs/baseline_submission.csv


In [ ]:
# Nueva URL de descarga directa del archivo ZIP de Google Drive
new_zip_file_url = 'https://drive.google.com/uc?id=1fd4f0MuMiuUT0IEZxfLONMF6gD4hGIND'
new_output_dir = './data_challenge_new/' # Directorio donde se extraerán los archivos

print(f"Descargando archivo ZIP desde: {new_zip_file_url}")
response = requests.get(new_zip_file_url)
response.raise_for_status() # Lanza una excepción si la descarga falla

# Crear el directorio de salida si no existe
os.makedirs(new_output_dir, exist_ok=True)

# Descomprimir el archivo en el directorio especificado
with zipfile.ZipFile(io.BytesIO(response.content)) as zf:
    zf.extractall(new_output_dir)

print(f"Archivos descomprimidos en: {new_output_dir}")
print("Listado de archivos extraídos del nuevo ZIP:")
for root, dirs, files in os.walk(new_output_dir):
    for name in files:
        print(os.path.join(root, name))
    for name in dirs:
        print(os.path.join(root, name) + '/')


Descargando archivo ZIP desde: https://drive.google.com/uc?id=1fd4f0MuMiuUT0IEZxfLONMF6gD4hGIND
Archivos descomprimidos en: ./data_challenge_new/
Listado de archivos extraídos del nuevo ZIP:
./data_challenge_new/datos_sinraw/
./data_challenge_new/datos_sinraw/test.csv
./data_challenge_new/datos_sinraw/train_sup.parquet
./data_challenge_new/datos_sinraw/train_sup.csv
./data_challenge_new/datos_sinraw/raw_signal_index.csv
./data_challenge_new/datos_sinraw/test.parquet
./data_challenge_new/datos_sinraw/train_unlabeled.csv
./data_challenge_new/datos_sinraw/train_unlabeled.parquet
./data_challenge_new/datos_sinraw/checksums.txt
./data_challenge_new/datos_sinraw/raw_signal_index.parquet
./data_challenge_new/datos_sinraw/data_dictionary.csv


In [ ]:
# Colab: descomentar. En local, si ya están instaladas, saltear.
!pip -q install lightgbm xgboost catboost pyarrow

import os, sys, warnings, itertools
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Carpeta con train_sup.parquet / test.parquet y con scoring.py del kit de la
# cátedra. Ajustar a donde estén montados los datos.
DATA = Path("data_challenge_new/datos_sinraw")
KIT  = Path("data_from_drive/participant_kit")

sys.path.insert(0, str(KIT))
sys.path.insert(0, str(KIT.parent))
import scoring
from scoring import compute_score, FAULT_IDS, LABEL_COLUMNS, FAMILIES

print("scoring cargado |", len(FAULT_IDS), "fallas |", sorted(set(FAMILIES.values())))


scoring cargado | 13 fallas | ['electrical', 'hydraulic', 'mechanical', 'structural']


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import sys

# Añadir el directorio del kit de participantes al path para que Python encuentre 'baseline'
sys.path.append('./data_from_drive/participant_kit/')

from baseline import (
    read_table, find_table, split_by_machine,
    estimate_prevalences, prevalence_submission,
)
from scoring import compute_score, validate_prediction_package, FAULT_IDS

# Ajustar DATA_DIR para que apunte a la carpeta donde se extrajeron train_sup.csv y test.csv
DATA_DIR = Path('./data_challenge_new/datos_sinraw/') # Ruta correcta de los datos
OUTPUT_DIR = Path('./outputs')

LABEL_COLS = [f'label_{f}' for f in FAULT_IDS]
SEVERITY_COLS = [f'severity_{f}' for f in FAULT_IDS]

In [ ]:
train = read_table(find_table(DATA_DIR, 'train_sup'))
test = read_table(find_table(DATA_DIR, 'test'))
print(f'train_sup: {train.shape}  (filas, columnas)')
print(f'test:      {test.shape}')
print(f'Máquinas en train: {train.machine_id.nunique()}')
print(f'Sesiones en train: {train.session_id.nunique()}')
train[['window_id', 'machine_id', 'session_id']].head()

train_sup: (1750, 77)  (filas, columnas)
test:      (1848, 49)
Máquinas en train: 26
Sesiones en train: 250


,window_id,machine_id,session_id
0,W0511a7920bdf7d480000,M0000022,S0511a7920bdf7d48
1,W0511a7920bdf7d480001,M0000022,S0511a7920bdf7d48
2,W0511a7920bdf7d480002,M0000022,S0511a7920bdf7d48
3,W0511a7920bdf7d480003,M0000022,S0511a7920bdf7d48
4,W0511a7920bdf7d480004,M0000022,S0511a7920bdf7d48


## 1. Carga y EDA rápido

La validación que interesa acá no es "¿hay nulos?" sino **¿la estructura del
target es la que dice la consigna?**. La consigna describe un problema
multilabel con estados combinados; los datos dicen otra cosa, y de esa
diferencia sale la decisión de modelado.

In [ ]:
train = pd.read_parquet(DATA / "train_sup.parquet")
test  = pd.read_parquet(DATA / "test.parquet")

print("train", train.shape, "| test", test.shape)
print("máquinas -> train %d, test %d, intersección %d"
      % (train.machine_id.nunique(), test.machine_id.nunique(),
         len(set(train.machine_id) & set(test.machine_id))))
print("sesiones -> train %d, test %d, intersección %d"
      % (train.session_id.nunique(), test.session_id.nunique(),
         len(set(train.session_id) & set(test.session_id))))

In [ ]:
# Limpieza: estas tablas vienen sintéticas y limpias. Se verifica, no se asume.
faltantes = train.isna().sum()
print("columnas con nulos:", int((faltantes > 0).sum()))
print("filas duplicadas   :", int(train.duplicated().sum()))
print("window_id únicos   :", train.window_id.is_unique, "|", test.window_id.is_unique)

constantes = [c for c in train.columns if train[c].nunique(dropna=False) <= 1]
print("columnas constantes:", constantes)

### Estructura del target

`is_combo` marca las ventanas con más de una falla simultánea. Si vale 0 en
todas las filas, el problema colapsa de 13 etiquetas independientes a **14
estados mutuamente excluyentes**, y conviene un softmax en vez de 13 binarios:
las probabilidades salen coherentes entre sí y suman ≤ 1, que es exactamente lo
que consume `choose_action` de la métrica.

In [ ]:
labels = train[LABEL_COLUMNS].to_numpy()
n_fallas = labels.sum(axis=1)

print("fallas por ventana:", pd.Series(n_fallas).value_counts().sort_index().to_dict())
print("is_combo.mean() = %.3f" % train.is_combo.mean())
print("E[#fallas] = %.3f  |  P(al menos una falla) = %.3f"
      % (n_fallas.mean(), (n_fallas > 0).mean()))

assert train.is_combo.sum() == 0, "hay combos: revisar la hipótesis de 14 estados"
assert set(np.unique(n_fallas)) <= {0, 1}, "alguna ventana tiene más de una falla"
print("\nconfirmado: cada ventana tiene exactamente 0 o 1 falla -> 14 estados")

# estado 0 = sano; estado k = falla F{k}
estado = np.where(n_fallas == 0, 0, labels.argmax(axis=1) + 1)
ESTADOS = ["sano"] + FAULT_IDS

### Prevalencia de las 14 clases

In [ ]:
prev = (pd.Series(estado).value_counts().sort_index()
        .rename(index=lambda k: ESTADOS[k]).to_frame("n"))
prev["prevalencia"] = prev.n / len(train)
prev["familia"] = ["sano"] + [FAMILIES[f] for f in FAULT_IDS]
display(prev)

fig, ax = plt.subplots(figsize=(10, 3.4))
colores = {"sano": "#2E6B45", "mechanical": "#0E4F5C", "structural": "#6B7A83",
           "electrical": "#9A5B00", "hydraulic": "#3D7B8F"}
ax.bar(prev.index, prev.prevalencia, color=[colores[f] for f in prev.familia])
ax.set_ylabel("prevalencia"); ax.set_title("Prevalencia por estado (train_sup)")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

print("severidad: de las ventanas CON falla, %.1f %% son severas (max_severity > 0.5)"
      % (100 * (train.loc[n_fallas > 0, [f"severity_{f}" for f in FAULT_IDS]].max(axis=1) > 0.5).mean()))

El desbalance es moderado (de 4.8 % a 9.6 % por falla, 12.4 % sano). No hace
falta remuestrear: lo que castiga la métrica no es el desbalance sino la
confusión **entre familias**, porque `cost_score` — que pesa 70 % — sólo
distingue las cuatro familias más el estado sano.

## 2. Mitigación del covariate shift

Train y test no comparten ninguna máquina. Cada máquina tiene su punto de
trabajo propio (rpm, temperatura, caudal), así que los valores **absolutos** de
los sensores identifican a la máquina más que al estado de salud. Un modelo
entrenado sobre valores absolutos aprende las 26 máquinas del train y no
generaliza a las 22 del test.

La corrección es expresar cada variable como desviación respecto del
comportamiento habitual de su propia máquina:

$$z_{i,c} = \frac{x_{i,c} - \mathrm{mediana}_{m(i)}(x_c)}{\mathrm{desvío}_{m(i)}(x_c) + \varepsilon}$$

Se usa **mediana** y no media por robustez ante las ventanas en falla, que son
justamente las colas de la distribución de cada máquina.

Esto es legítimo sobre el test: usa sólo las columnas de features y el
`machine_id`, nunca las etiquetas. Se calcula por separado en cada tabla.

In [ ]:
EXCLUIR = set(["window_id", "machine_id", "session_id", "is_normal", "is_combo"]
              + LABEL_COLUMNS + [f"severity_{f}" for f in FAULT_IDS])
FEATS = [c for c in train.columns if c not in EXCLUIR]
print("features tabulares originales:", len(FEATS))

def zscore_por_maquina(df, cols):
    """Desviación respecto de la propia máquina. No mira etiquetas."""
    g = df.groupby("machine_id")[cols]
    z = (df[cols] - g.transform("median")) / (g.transform("std") + 1e-9)
    z.columns = [c + "__z" for c in cols]
    return z

def construir_X(df):
    return pd.concat([df[FEATS], zscore_por_maquina(df, FEATS)], axis=1)

X_train, X_test = construir_X(train), construir_X(test)
print("matriz final:", X_train.shape, "(originales + z-score)")

### Verificación gráfica del alineamiento

Un KDE de train contra test, antes y después. Si la normalización funciona, las
dos curvas se superponen en el panel derecho.

In [ ]:
from scipy.stats import gaussian_kde, ks_2samp

CRITICAS = ["rpm_mean", "voltage_a__rms", "flow_mean", "acc_radial_b__crest"]

def kde(ax, a, b, titulo):
    grid = np.linspace(min(a.min(), b.min()), max(a.max(), b.max()), 200)
    ax.fill_between(grid, gaussian_kde(a)(grid), color="#0E4F5C", alpha=.35, label="train")
    ax.fill_between(grid, gaussian_kde(b)(grid), color="#9A5B00", alpha=.35, label="test")
    ax.set_title(titulo, fontsize=10)
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_yticks([])

fig, axes = plt.subplots(len(CRITICAS), 2, figsize=(11, 2.5 * len(CRITICAS)))
for fila, c in enumerate(CRITICAS):
    ks_c = ks_2samp(train[c], test[c]).statistic
    ks_z = ks_2samp(X_train[c + "__z"].dropna(), X_test[c + "__z"].dropna()).statistic
    kde(axes[fila, 0], train[c].values, test[c].values, f"{c} — crudo (KS={ks_c:.3f})")
    kde(axes[fila, 1], X_train[c + "__z"].dropna().values, X_test[c + "__z"].dropna().values,
        f"{c} — z-score por máquina (KS={ks_z:.3f})")
axes[0, 0].legend(fontsize=8, frameon=False)
plt.tight_layout(); plt.show()

In [ ]:
# Efecto sobre las 46 variables, no sólo sobre las cuatro del gráfico.
ks_crudo = [ks_2samp(train[c], test[c]).statistic for c in FEATS]
ks_norm  = [ks_2samp(X_train[c + "__z"].dropna(), X_test[c + "__z"].dropna()).statistic
            for c in FEATS]

print("KS medio train vs test:  crudo = %.3f   ->   z-score = %.3f"
      % (np.mean(ks_crudo), np.mean(ks_norm)))
print("variables con KS > 0.2:  crudo = %d      ->   z-score = %d"
      % (sum(k > .2 for k in ks_crudo), sum(k > .2 for k in ks_norm)))

Las 11 variables que estaban claramente desalineadas quedan en cero, y el KS
medio cae a menos de la mitad. Se **conservan las dos versiones** de cada
variable: los valores absolutos siguen teniendo información física real
(una temperatura de devanado alta es alta en cualquier máquina) y los árboles
eligen cuál usar.

## 3. Benchmark multimodelo

Seis algoritmos con la misma partición, las mismas features y la métrica
oficial. La regla del equipo: **ninguna diferencia se acepta si no supera el
desvío entre folds**, así que se reporta ese desvío junto a cada resultado.

La partición es `StratifiedGroupKFold(5)` agrupando por `machine_id`:
estratifica por estado para que las fallas raras aparezcan en cada fold, y
mantiene cada máquina entera de un solo lado. Es la única partición que imita
la relación real entre train y test, donde las máquinas son disjuntas.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

MODELOS = {
    "LightGBM": lambda: lgb.LGBMClassifier(
        objective="multiclass", num_class=14, n_estimators=600, learning_rate=0.05,
        num_leaves=31, min_child_samples=20, subsample=0.9, subsample_freq=1,
        colsample_bytree=0.8, reg_lambda=1.0, verbose=-1, random_state=RANDOM_STATE),
    "XGBoost": lambda: xgb.XGBClassifier(
        objective="multi:softprob", num_class=14, n_estimators=600, learning_rate=0.05,
        max_depth=6, min_child_weight=3, subsample=0.9, colsample_bytree=0.8,
        reg_lambda=1.0, tree_method="hist", verbosity=0, random_state=RANDOM_STATE),
    "CatBoost": lambda: CatBoostClassifier(
        loss_function="MultiClass", iterations=600, learning_rate=0.05, depth=6,
        l2_leaf_reg=3.0, random_seed=RANDOM_STATE, verbose=0, allow_writing_files=False),
    "RandomForest": lambda: make_pipeline(
        SimpleImputer(strategy="median"),
        RandomForestClassifier(n_estimators=600, min_samples_leaf=2, max_features="sqrt",
                               n_jobs=-1, random_state=RANDOM_STATE)),
    "ExtraTrees": lambda: make_pipeline(
        SimpleImputer(strategy="median"),
        ExtraTreesClassifier(n_estimators=600, min_samples_leaf=2, max_features="sqrt",
                             n_jobs=-1, random_state=RANDOM_STATE)),
    "RegLogistica": lambda: make_pipeline(
        SimpleImputer(strategy="median"), StandardScaler(),
        LogisticRegression(max_iter=2000, C=0.5, random_state=RANDOM_STATE)),
}

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
FOLDS = list(cv.split(X_train, estado, train.machine_id.to_numpy()))
for k, (a, b) in enumerate(FOLDS):
    print("fold %d: %4d ajuste / %3d validación | máquinas de validación: %d"
          % (k, len(a), len(b), train.machine_id.iloc[b].nunique()))

### La métrica oficial como único juez

`compute_score` necesita el DataFrame de verdad completo (etiquetas y
severidades) más una entrega con `window_id` y las 13 probabilidades. La función
de abajo arma esa entrega desde una matriz OOF y devuelve las tres componentes.

Se evalúa también **por fold**, para tener el desvío que decide si una mejora es
real o ruido.

In [ ]:
FAM_ORDEN = sorted(set(FAMILIES.values()))

def evaluar(proba13, idx=None):
    """proba13: (n, 13) alineado con `train`. idx: subconjunto de filas."""
    idx = np.arange(len(train)) if idx is None else idx
    entrega = pd.DataFrame({"window_id": train.window_id.iloc[idx].to_numpy()})
    for j, f in enumerate(FAULT_IDS):
        entrega[f] = np.clip(proba13[idx][:, j], 0, 1)
    return compute_score(train.iloc[idx], entrega)["overall"]

def familia_de(e):
    return "sano" if e == 0 else FAMILIES[FAULT_IDS[e - 1]]

def correr(nombre, ctor):
    oof = np.zeros((len(X_train), 14))
    for a, b in FOLDS:
        modelo = ctor()
        modelo.fit(X_train.iloc[a], estado[a])
        oof[b] = modelo.predict_proba(X_train.iloc[b])
    r = evaluar(oof[:, 1:])
    por_fold = [evaluar(oof[:, 1:], b)["final_score"] for _, b in FOLDS]
    fam_real = np.array([familia_de(e) for e in estado])
    fam_pred = np.array([familia_de(e) for e in oof.argmax(1)])
    return oof, dict(
        modelo=nombre, final=r["final_score"], cost=r["cost_score"],
        prob=r["prob_score"], diag=r["diag_score"], brier=r["brier"],
        macro_f1=r["macro_f1"], acc_estado=(oof.argmax(1) == estado).mean(),
        acc_familia=(fam_real == fam_pred).mean(), sd_folds=float(np.std(por_fold)))

In [ ]:
oofs, filas = {}, []
for nombre, ctor in MODELOS.items():
    oofs[nombre], fila = correr(nombre, ctor)
    filas.append(fila)
    print("  %-13s final = %6.2f   (sd entre folds = %.2f)"
          % (nombre, fila["final"], fila["sd_folds"]))

### Referencias

Sin un piso y un techo, los números de arriba no significan nada. El piso es el
baseline oficial de la cátedra (prevalencia constante). El techo es el oráculo:
lo que daría el modelo perfecto.

In [ ]:
# Piso: P0, la prevalencia observada repetida en todas las filas.
P0 = np.tile(train[LABEL_COLUMNS].mean().to_numpy(), (len(train), 1))
r_p0 = evaluar(P0)

# Techo: las etiquetas exactas.
r_or = evaluar(train[LABEL_COLUMNS].to_numpy(dtype=float))

print("P0 prevalencia   final = %6.2f  (cost %5.2f)" % (r_p0["final_score"], r_p0["cost_score"]))
print("oráculo perfecto final = %6.2f  (cost %5.2f)" % (r_or["final_score"], r_or["cost_score"]))
print("""
El oráculo no llega a 100: aun conociendo la falla exacta hay que pagar la
inspección de $4. Con naive_cost = %.2f y costo mínimo alcanzable
0.876 x 4 = %.2f, cost_score se planta en %.2f.""" %
      (r_or["naive_cost"], 0.876 * 4, r_or["cost_score"]))

## 4. Análisis de resultados

In [ ]:
tabla = (pd.DataFrame(filas)
         .sort_values("final", ascending=False)
         .reset_index(drop=True))
tabla.insert(2, "vs_P0", tabla.final - r_p0["final_score"])
tabla.insert(3, "pct_techo", 100 * tabla.final / r_or["final_score"])
display(tabla.style.format({
    "final": "{:.2f}", "vs_P0": "+{:.2f}", "pct_techo": "{:.1f}%", "cost": "{:.2f}",
    "prob": "{:.2f}", "diag": "{:.2f}", "brier": "{:.4f}", "macro_f1": "{:.3f}",
    "acc_estado": "{:.3f}", "acc_familia": "{:.3f}", "sd_folds": "{:.2f}"})
    .background_gradient(subset=["final"], cmap="BuGn"))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.6))
orden = tabla.sort_values("final")
ax.barh(orden.modelo, orden.final, xerr=orden.sd_folds, color="#0E4F5C",
        error_kw=dict(ecolor="#9A5B00", capsize=3, lw=1.2))
ax.axvline(r_p0["final_score"], color="#8F2A1D", ls="--", lw=1.2)
ax.text(r_p0["final_score"], -0.6, " P0 = %.1f" % r_p0["final_score"],
        color="#8F2A1D", fontsize=9, va="top")
ax.axvline(r_or["final_score"], color="#2E6B45", ls=":", lw=1.2)
ax.text(r_or["final_score"], -0.6, " oráculo = %.1f" % r_or["final_score"],
        color="#2E6B45", fontsize=9, va="top", ha="right")
ax.set_xlabel("final_score (barra de error: desvío entre folds)")
ax.set_xlim(0, r_or["final_score"] * 1.05)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

### El spread entre algoritmos es menor que el ruido

Ese gráfico es el resultado principal del notebook, y no es el que se esperaba.
Las barras de error se solapan casi por completo: la distancia entre el primero
y el cuarto es del orden del desvío entre folds de cualquiera de ellos.
**Cambiar de algoritmo no mueve la aguja.** Los seis, incluida una regresión
logística, aterrizan en la misma banda.

Eso es la definición operativa de un techo: cuando modelos con sesgos
inductivos tan distintos convergen al mismo número, el límite no está en el
algoritmo sino en la **información disponible en las features**.

Vale la pena mirar la descomposición, porque los modelos no fallan igual.

In [ ]:
comp = tabla.set_index("modelo")[["cost", "prob", "diag"]]
fig, axes = plt.subplots(1, 3, figsize=(12, 3.2), sharey=True)
for ax, col, color, peso in zip(axes, comp.columns,
                                ["#0E4F5C", "#9A5B00", "#3D7B8F"], ["70 %", "20 %", "10 %"]):
    ax.barh(comp.index, comp[col], color=color)
    ax.set_title("%s_score  (peso %s)" % (col, peso), fontsize=10)
    ax.spines[["top", "right"]].set_visible(False)
    ax.invert_yaxis()
plt.tight_layout(); plt.show()

print(comp.to_string(float_format=lambda v: "%.2f" % v))

Los boosters (LightGBM, XGBoost) ganan en `diag_score`: discriminan mejor y
empujan más ventanas por encima del umbral de 0.5 del macro-F1. Los bagging
(RandomForest, ExtraTrees) y CatBoost ganan en `prob_score`: promedian muchos
árboles y salen mejor calibrados, con Brier más bajo.

Nadie gana en las tres. **Eso es una invitación al ensamble**, no a elegir un
ganador: hay diversidad real que explotar.

In [ ]:
def ensamblar(nombres):
    P = np.mean([oofs[n] for n in nombres], axis=0)
    return P / P.sum(axis=1, keepdims=True)

CANDIDATOS = {
    "LightGBM solo (mejor individual)": ["LightGBM"],
    "LGB + XGB":                        ["LightGBM", "XGBoost"],
    "LGB + XGB + Cat":                  ["LightGBM", "XGBoost", "CatBoost"],
    "LGB + XGB + Cat + RF":             ["LightGBM", "XGBoost", "CatBoost", "RandomForest"],
    "LGB + RF + RegLogistica":          ["LightGBM", "RandomForest", "RegLogistica"],
}

ens = []
for nombre, combo in CANDIDATOS.items():
    P = ensamblar(combo)
    r = evaluar(P[:, 1:])
    por_fold = [evaluar(P[:, 1:], b)["final_score"] for _, b in FOLDS]
    ens.append(dict(ensamble=nombre, final=r["final_score"], cost=r["cost_score"],
                    prob=r["prob_score"], diag=r["diag_score"],
                    sd_folds=float(np.std(por_fold)),
                    por_fold=" ".join("%.1f" % v for v in por_fold)))
display(pd.DataFrame(ens).style.format({
    "final": "{:.2f}", "cost": "{:.2f}", "prob": "{:.2f}",
    "diag": "{:.2f}", "sd_folds": "{:.2f}"}).hide(axis="index"))

El promedio simple de probabilidades da entre +0.6 y +2.2 sobre el mejor
individual, y **gana fold por fold**, no sólo en el agregado. Esa consistencia
es la que distingue una mejora real del ruido de partición.

Una advertencia honesta sobre el último candidato: `LGB + RF + RegLogistica`
salió de buscar exhaustivamente entre las 41 combinaciones posibles sobre esta
misma partición. Ganar en los 5 folds es evidencia fuerte, pero **la búsqueda
sobre el OOF infla la estimación** — es la trampa de sobreajuste al OOF que
figura en el kit del equipo. La celda siguiente es la que lo dirime.

In [ ]:
# Confirmación con OTRA partición. Si la ventaja sobrevive a un cambio de
# semilla de folds, es real; si se evapora, era selección sobre el OOF.
SEMILLA_CONTROL = 2024
folds_ctrl = list(StratifiedGroupKFold(5, shuffle=True, random_state=SEMILLA_CONTROL)
                  .split(X_train, estado, train.machine_id.to_numpy()))

def correr_con(nombre, ctor, folds):
    oof = np.zeros((len(X_train), 14))
    for a, b in folds:
        m = ctor(); m.fit(X_train.iloc[a], estado[a])
        oof[b] = m.predict_proba(X_train.iloc[b])
    return oof

oofs_ctrl = {n: correr_con(n, MODELOS[n], folds_ctrl)
             for n in ["LightGBM", "XGBoost", "CatBoost", "RandomForest", "RegLogistica"]}

print("%-34s %10s %10s" % ("candidato", "semilla 42", "semilla 2024"))
for nombre, combo in CANDIDATOS.items():
    a = evaluar(ensamblar(combo)[:, 1:])["final_score"]
    Pc = np.mean([oofs_ctrl[n] for n in combo], axis=0)
    Pc = Pc / Pc.sum(axis=1, keepdims=True)
    b = evaluar(Pc[:, 1:])["final_score"]
    print("%-34s %10.2f %10.2f" % (nombre, a, b))

### Resultado de la confirmación

La partición sola mueve más que el algoritmo. Con la semilla de control el
ranking individual **se reordena por completo**: RandomForest pasa a primero
(45.79) y CatBoost cae a último (44.26), cuando con la semilla 42 estaban
cuarto y tercero. LightGBM baja de 47.17 a 45.62 sin que nada del modelo haya
cambiado.

| Candidato | semilla 42 | semilla 2024 | ventaja sobre LightGBM solo |
|---|---|---|---|
| LightGBM solo | 47.17 | 45.62 | — |
| LGB + XGB | 47.77 | 45.85 | +0.61 / +0.23 |
| LGB + XGB + Cat | 47.85 | 46.84 | +0.69 / +1.22 |
| LGB + XGB + Cat + RF | 47.87 | 47.29 | +0.70 / +1.67 |
| **LGB + RF + RegLogistica** | **49.36** | **47.65** | **+2.19 / +2.03** |

Los cuatro ensambles mejoran al mejor individual en **las dos particiones**. Y
el candidato sospechoso —el que había salido de buscar entre 41 combinaciones—
mantiene su ventaja casi intacta al cambiar de partición: +2.19 y +2.03. Eso
descarta el sobreajuste de selección: si la ventaja hubiera venido de elegir
sobre el OOF, se habría evaporado acá.

La lectura de por qué funciona: la regresión logística es el peor modelo
individual de los seis, pero es el único **lineal**. Sus errores están
descorrelacionados de los de los árboles, y en un promedio eso vale más que su
desempeño propio. Es el argumento clásico a favor de la diversidad por sobre la
calidad individual en un ensamble.

### Conclusión: dónde está el techo tabular y con qué seguimos

**El techo con datos exclusivamente tabulares está en torno a 48 puntos**
(48.50 promediando las dos particiones, con un mejor caso de 49.36), contra un
piso de 29.05 del baseline P0 y un techo teórico de 73.96 del oráculo. Estamos
en unos dos tercios de lo alcanzable.

La evidencia de que es un techo real y no una limitación de nuestra elección de
modelo es doble:

- **Seis algoritmos con sesgos inductivos muy distintos convergen** dentro de un
  margen de 4.5 puntos, más chico que el ruido de partición de varios de ellos.
- **El ranking entre algoritmos no es estable**: al cambiar la semilla de folds
  se reordena por completo. Un ranking que se da vuelta con la partición no
  está midiendo calidad de algoritmo, está midiendo ruido.

Consecuencia práctica para el reparto de la tarde: **buscar un algoritmo mejor
es tiempo perdido**, y el tuneo de hiperparámetros todavía más, porque mueve
menos que el cambio de familia de modelo, que ya vimos que no mueve nada.

Con qué avanzar, en orden:

1. **Ensamble por promedio simple, con diversidad deliberada.** LightGBM +
   RandomForest + regresión logística da +2 puntos consistentes en las dos
   particiones. Sale de modelos que ya están entrenados: es la mejora más
   barata que queda sobre la mesa. Incluir el modelo lineal aunque sea el peor
   individual.
2. **Calibración explícita.** `prob_score` pesa 20 % y está en 80–83;
   `diag_score` pesa 10 % y está en 36, que es donde más margen relativo queda.
   Ninguno de los dos requiere features nuevas.
3. **Las señales crudas son la única vía para romper 50.** Los descriptores
   tabulares traen un solo armónico por canal (1x y 2x), y las frecuencias que
   separan las fallas de rodamiento entre sí (BPFO, BPFI, BSF) no están en la
   tabla. La confusión intra-familia mecánica —seis de las trece fallas— no se
   resuelve con lo que hay acá. La accuracy de familia se estanca en 0.60–0.64
   en los seis modelos, y `cost_score` es exactamente lo que paga por ella.

Lo que este notebook deja listo para la tarde: la matriz de features
normalizada, la partición agrupada, la función de evaluación con la métrica
oficial y un punto de comparación honesto para decidir si cualquier feature
nueva de los NPZ aporta o no.

### Registro para el probatorio

Las decisiones tomadas acá, con su evidencia, para que no haya que
reconstruirlas el viernes a las 09:30.

In [ ]:
REGISTRO = [
    dict(tipo="hallazgo", texto="El problema no es multilabel",
         evidencia="is_combo = 0 en las 1750 filas; E[#fallas] = P(falla) = 0.876",
         decision="Softmax de 14 estados en vez de 13 binarios independientes"),
    dict(tipo="hallazgo", texto="Train y test no comparten máquinas",
         evidencia="intersección de machine_id vacía; KS medio 0.139 entre las tablas",
         decision="z-score por machine_id + StratifiedGroupKFold agrupado"),
    dict(tipo="decision", texto="Normalización por máquina",
         evidencia="KS medio 0.139 -> 0.059; variables con KS>0.2: 11 -> 0",
         decision="Se conservan features originales y normalizadas"),
    dict(tipo="hallazgo", texto="El algoritmo no es la palanca",
         evidencia="6 modelos entre 42.7 y 47.2 con sd entre folds de 1.8 a 6.2",
         decision="Se abandona la búsqueda de modelo; se pasa a ensamble y calibración"),
    dict(tipo="descarte", texto="Tuneo de hiperparámetros en esta etapa",
         evidencia="el cambio de familia de modelo mueve menos que el ruido de partición",
         decision="no se invierte tiempo hasta tener features nuevas"),
]
registro = pd.DataFrame(REGISTRO)
display(registro)
registro.to_json("registro_benchmark.json", orient="records", force_ascii=False, indent=2)
print("guardado en registro_benchmark.json")